In [1]:
%pip install matplotlib
%pip install pandas
%pip install prophet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from prophet import Prophet
import pandas as pd
import sqlite3
import os

# Define absolute database path
DB_PATH = r"E:\genai-hackathon\SmartRetail_AI\database\retail.db"

def check_db_exists():
    """Check if the database file exists before proceeding."""
    if not os.path.exists(DB_PATH):
        raise FileNotFoundError(f"Error: Database file not found at {DB_PATH}")

def load_data():
    """Load historical sales data from the SQLite database."""
    check_db_exists()
    
    try:
        conn = sqlite3.connect(DB_PATH)
        query = 'SELECT Date, "Sales Quantity" FROM DemandForecasting'
        df = pd.read_sql(query, conn)
        conn.close()
        
        if df.empty:
            raise ValueError("Warning: The DemandForecasting table is empty.")
        
        df.rename(columns={"Date": "ds", "SalesQuantity": "y"}, inplace=True)
        df["ds"] = pd.to_datetime(df["ds"])
        return df

    except Exception as e:
        print(f"Error while loading data: {e}")
        return None

def train_prophet_model(df):
    """Train a Prophet model on the provided data."""
    try:
        model = Prophet()
        model.fit(df)
        return model
    except Exception as e:
        print(f"Error while training model: {e}")
        return None

def forecast_demand(model, periods=30):
    """Generate future demand predictions."""
    try:
        future = model.make_future_dataframe(periods=periods)
        forecast = model.predict(future)
        return forecast
    except Exception as e:
        print(f"Error while forecasting: {e}")
        return None

def save_forecast_to_db(forecast):
    """Save forecasted data to SQLite database."""
    check_db_exists()
    
    try:
        conn = sqlite3.connect(DB_PATH)
        forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_sql(
            'ForecastedDemand', conn, if_exists='replace', index=False
        )
        conn.close()
        print("✅ Forecasting results successfully saved to database.")
    except Exception as e:
        print(f"Error while saving forecast: {e}")

def main():
    df = load_data()
    if df is None:
        return

    model = train_prophet_model(df)
    if model is None:
        return

    forecast = forecast_demand(model)
    if forecast is None:
        return

    save_forecast_to_db(forecast)

if __name__ == "__main__":
    main()


Error while training model: Dataframe must have columns "ds" and "y" with the dates and values respectively.
